# FCD SynthSeg — 7-Class Lesion Segmentation

**Task:** Focal Cortical Dysplasia (FCD Type II) lesion segmentation from 3-D FLAIR MRI.  
**Approach:** SynthSeg-style training — synthetic images are generated on-the-fly from label maps using a per-class Gaussian Mixture Model. Subject-specific FCD appearance augmentations are injected into the synthetic branch. The network trains exclusively on synthetic images and is evaluated on real FLAIR scans.

**7 output classes:**

| Class | Tissue |
|-------|--------|
| 0 | Background |
| 1 | White Matter |
| 2 | Cerebral Cortex |
| 3 | Deep Gray Matter |
| 4 | CSF |
| 5 | WM-GM Separator (label 18) |
| 6 | FCD Lesion |

**Pipeline:**
```
labelmap.nii  →  one-hot (19, D, H, W)
              →  PerClassGMM           →  synthetic FLAIR  [~0, 255]
              →  FCDAugmentations      →  FCD appearance injected
              →  IntensityTransform    →  normalized        [0, 1]
              →  label fusion          →  ROI voxels → class 6
              →  SegNet forward
              →  DiceCE loss  →  backward (synthetic branch only)
Real FLAIR passes through the network forward-only for monitoring (no gradient).
```

---
## § 1 — Configuration

All run-level settings live here. Edit before launching.

| Variable | Purpose |
|----------|---------|
| `TRAINING_TIME_MINUTES` | Hard time limit passed to `TimeLimitCallback` |
| `RUN_NAME` | Experiment subdirectory under `experiments/` |
| `INPUT` | Kaggle input dataset path containing previous `experiments/` output |
| `REPO_URL` / `BRANCH_NAME` | Git source for `learn2synth` and training script |
| `CLEAR_IMAGES_ON_START` | Wipe `images/` on each run to save storage |

In [ ]:
# ──────────────────── Training ───────────────────────────────────────────────
TRAINING_TIME_MINUTES = (11 * 60) + 45          # 11 h 45 min

# ──────────────────── Experiment ─────────────────────────────────────────────
RUN_NAME              = "fcd_experiment_run"     # outputs → experiments/<RUN_NAME>/

# ──────────────────── Input Dataset ──────────────────────────────────────────
INPUT                 = ""                       # path to Kaggle input dataset

# ──────────────────── Repository ─────────────────────────────────────────────
REPO_URL              = "https://github.com/YassienTawfikk/SynthFCD.git"
BRANCH_NAME           = ""                       # e.g. "version-54"

# ──────────────────── Flags ───────────────────────────────────────────────────
CLEAR_IMAGES_ON_START = True                     # wipe images/ to save storage

---
## § 2 — Environment Setup

Pinned dependencies for reproducibility on Kaggle P100/T4 instances.

| Package | Reason |
|---------|--------|
| `cornucopia` | SynthSeg synthesis transforms — pinned commit for API stability |
| `torch==2.6.0+cu124` | CUDA 12.4 alignment |
| `numpy<2.0` | Avoids `_ARRAY_FUNCTION_DISPATCH` ABI breakage |
| `torchmetrics` | `DiceScore` metric for validation |
| `pytorch-lightning` | Training loop, checkpointing, CLI |

> ⚠️ Do not upgrade `cornucopia` past the pinned commit — the synthesis API is not stable.

In [ ]:
import os
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
os.environ['PYDEVD_WARN_SLOW_RESOLVE_TIMEOUT'] = '0'
os.environ['PYTHONFROZENMODULES'] = 'off'

In [ ]:
# ── 1. Cornucopia (pinned commit for stability) ──────────────────────────────
!pip install -qq git+https://github.com/balbasty/cornucopia@6f8ab58dfcfe8978c9aa9e8b05898dcf7d75bb5b > /dev/null 2>&1

# ── 2. Core dependencies ─────────────────────────────────────────────────────
!pip install -qq -U        "jsonargparse[signatures]>=4.27.7" > /dev/null 2>&1


# ── 3. PyTorch stack — GPU P100 (CUDA 12.4) ──────────────────────────────────
!pip install -qq --no-deps "protobuf==3.20.3"                 > /dev/null 2>&1
!pip install -qq "torch==2.6.0+cu124" "torchvision==0.21.0+cu124" "torchaudio==2.6.0+cu124" \
               --index-url https://download.pytorch.org/whl/cu124 > /dev/null 2>&1

# ── 4. NumPy < 2.0 (fixes _ARRAY_API compatibility issues) ───────────────────
!pip install -qq "numpy<2.0" > /dev/null 2>&1

# ── 5. Scientific + visualization stack ──────────────────────────────────────
!pip install -qq pandas matplotlib scipy monai > /dev/null 2>&1

print("Dependencies installed.")

---
## § 3 — Repository & Package Setup

Clones `SynthFCD` from GitHub (or pulls latest changes if already present), copies `learn2synth` and `scripts` into the working directory, then removes the cloned repo to save storage.

In [ ]:
import shutil
import csv
import subprocess
import datetime

# ── Paths ─────────────────────────────────────────────────────────────────────
WORKING_DIR     = "/kaggle/working"
CLONED_REPO_DIR = os.path.join(WORKING_DIR, "SynthFCD")
PACKAGE_DIR     = os.path.join(WORKING_DIR, "learn2synth")
STATS_DIR       = os.path.join(WORKING_DIR, "flair-stats-synthseg")
SCRIPTS_DIR     = os.path.join(WORKING_DIR, "scripts")

os.makedirs(STATS_DIR,   exist_ok=True)
os.makedirs(SCRIPTS_DIR, exist_ok=True)

# ── Clone or pull ─────────────────────────────────────────────────────────────
if not os.path.exists(CLONED_REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH_NAME, REPO_URL, CLONED_REPO_DIR],
                   check=True, capture_output=True)
else:
    subprocess.run(["git", "-C", CLONED_REPO_DIR, "fetch",    "origin"],              check=True, capture_output=True)
    subprocess.run(["git", "-C", CLONED_REPO_DIR, "checkout",  BRANCH_NAME],          check=True, capture_output=True)
    subprocess.run(["git", "-C", CLONED_REPO_DIR, "pull",     "origin", BRANCH_NAME], check=True, capture_output=True)

SOURCE_ROOT = CLONED_REPO_DIR

# ── Copy learn2synth ──────────────────────────────────────────────────────────
INPUT_SOURCE = os.path.join(SOURCE_ROOT, "learn2synth")

if os.path.exists(PACKAGE_DIR):
    shutil.rmtree(PACKAGE_DIR)

if os.path.exists(INPUT_SOURCE):
    shutil.copytree(INPUT_SOURCE, PACKAGE_DIR)
else:
    raise FileNotFoundError(f"learn2synth not found at: {INPUT_SOURCE}")

# ── Copy flair_stats & scripts ────────────────────────────────────────────────
for source_dir in [os.path.join(SOURCE_ROOT, "flair_stats"), os.path.join(SOURCE_ROOT, "scripts")]:
    if not os.path.exists(source_dir):
        print(f"[Setup] Directory not found: {source_dir}")
        continue

    destination_dir = SCRIPTS_DIR if os.path.basename(source_dir) == "scripts" else STATS_DIR

    for item in os.listdir(source_dir):
        src = os.path.join(source_dir, item)
        dst = os.path.join(destination_dir, item)
        shutil.copytree(src, dst, dirs_exist_ok=True) if os.path.isdir(src) else shutil.copy2(src, dst)

# ── Cleanup cloned repo ───────────────────────────────────────────────────────
if os.path.exists(CLONED_REPO_DIR):
    shutil.rmtree(CLONED_REPO_DIR)

print("Repository setup complete.")

---
## § 4 — Pre-Launch Setup

Restores the experiment directory, resolves the resume checkpoint, preserves metric history, and cleans up redundant files before training starts.

### 4a — Restore Run Directory

Copies the previous run directory from the Kaggle input dataset into `/kaggle/working/experiments/`. If no previous run exists, creates a fresh directory.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
EXPERIMENTS_SRC = f"{INPUT}/experiments"
EXPERIMENTS_DIR = os.path.join(WORKING_DIR, "experiments")

SRC_RUN_DIR = os.path.join(EXPERIMENTS_SRC, RUN_NAME)
RUN_DIR     = os.path.join(EXPERIMENTS_DIR, RUN_NAME)

# ── Restore or initialize ─────────────────────────────────────────────────────
if os.path.exists(SRC_RUN_DIR):
    shutil.copytree(SRC_RUN_DIR, RUN_DIR, dirs_exist_ok=True)
else:
    os.makedirs(RUN_DIR, exist_ok=True)

# ── Images directory ──────────────────────────────────────────────────────────
images_dir = os.path.join(RUN_DIR, "images")

if CLEAR_IMAGES_ON_START and os.path.exists(images_dir):
    shutil.rmtree(images_dir)

os.makedirs(images_dir, exist_ok=True)

### 4b — Checkpoint Resolver

Resolves which checkpoint to resume from.

| Priority | File | Source |
|----------|------|--------|
| 1 | `last.ckpt` | `ModelCheckpoint(save_last=True)` — written every epoch |
| 2 | None | Fresh start |

In [ ]:
CKPT_DIR  = os.path.join(SRC_RUN_DIR, "checkpoints")
LAST_CKPT = os.path.join(CKPT_DIR, "last.ckpt")

if os.path.exists(LAST_CKPT):
    CKPT_PATH   = LAST_CKPT
    ckpt_source = "last.ckpt"
else:
    CKPT_PATH   = None
    ckpt_source = "fresh start"

print(f"Checkpoint : {ckpt_source}")
if CKPT_PATH:
    print(f"Path       : {CKPT_PATH}")

### 4c — Metric History

On each resume, Lightning overwrites `metrics.csv` with only the current run's data. To preserve the full training history across runs:

1. **Before training** — merge `metrics.csv` into `metrics_history.csv`
2. **After training** — merge `metrics_history.csv` + new `metrics.csv` → single `metrics.csv`, then delete `metrics_history.csv`

This keeps the output dataset clean: only `metrics.csv` (complete history) is needed as input for the next run.

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def _is_value_present(value):
    """Return True if a CSV cell has a real value."""
    if value is None:
        return False
    return str(value).strip() not in ("", "nan", "NaN", "None")


def _safe_number(value, default=float("inf")):
    """Convert CSV string values to numbers for stable sorting."""
    try:
        return float(value) if _is_value_present(value) else default
    except Exception:
        return default


def read_metric_csv(csv_path):
    """Read a Lightning metrics CSV using only the standard library."""
    if not os.path.exists(csv_path):
        return [], []
    with open(csv_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        fieldnames = list(reader.fieldnames or [])
    return fieldnames, rows


def write_metric_csv(csv_path, rows, fieldnames):
    """Write merged metric rows using a stable union of all CSV columns."""
    if not rows:
        return
    all_columns = list(fieldnames)
    for row in rows:
        for col in row:
            if col not in all_columns:
                all_columns.append(col)
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=all_columns)
        writer.writeheader()
        writer.writerows(rows)


def combine_metric_rows(rows):
    """
    Merge duplicate Lightning metric rows while keeping non-empty values.

    Lightning often writes different metrics on separate rows for the same
    epoch/step. This combines those rows safely without pandas.
    """
    if not rows:
        return []
    merged = {}
    for row in rows:
        key = (row.get("epoch", ""), row.get("step", ""))
        if key not in merged:
            merged[key] = dict(row)
        else:
            for col, value in row.items():
                if not _is_value_present(merged[key].get(col)) and _is_value_present(value):
                    merged[key][col] = value
    return sorted(
        merged.values(),
        key=lambda r: (_safe_number(r.get("epoch")), _safe_number(r.get("step"))),
    )

In [ ]:
# ── Merge metrics.csv → metrics_history.csv before training overwrites it ─────
metrics_path = os.path.join(RUN_DIR, "metrics.csv")
history_path = os.path.join(RUN_DIR, "metrics_history.csv")

if os.path.exists(metrics_path):
    history_fields, history_rows = read_metric_csv(history_path)
    metrics_fields, metrics_rows = read_metric_csv(metrics_path)

    all_rows   = history_rows + metrics_rows
    fieldnames = history_fields + [c for c in metrics_fields if c not in history_fields]
    merged     = combine_metric_rows(all_rows)

    write_metric_csv(history_path, merged, fieldnames)

### 4d — Cleanup Redundant Files

Removes files that accumulate across runs and are no longer needed:
- `metrics_before_resume_*.csv` — pre-merge snapshots (already merged into history)
- `events.out.tfevents.*` — TensorBoard event files (one created per run)

In [ ]:
if os.path.isdir(RUN_DIR):
    for fname in os.listdir(RUN_DIR):
        if fname.startswith("metrics_before_resume_") and fname.endswith(".csv"):
            os.remove(os.path.join(RUN_DIR, fname))
        elif fname.startswith("events.out.tfevents."):
            os.remove(os.path.join(RUN_DIR, fname))

---
## § 5 — Launch

Starts or resumes training. All hyperparameters are passed as CLI flags.

**Key flags:**

| Flag | Value | Notes |
|------|-------|-------|
| `--model.flair_modality` | `true` | Required for the FLAIR GMM synthesis path |
| `--data.eval` | `0.2` | 20% of raw subjects held out for validation (~12 subjects) |
| `--data.split_seed` | `42` | Reproducible train/val split |
| `--trainer.precision` | `16-mixed` | AMP; synthesis runs under `autocast(enabled=False)` |
| `--seed_everything` | `0` | Global seed for reproducibility |

In [ ]:
ckpt_arg = f'--ckpt_path "{CKPT_PATH}"' if CKPT_PATH else ""

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
  L2S_RUN_NAME={RUN_NAME} \
  L2S_TIME_LIMIT_MINUTES={TRAINING_TIME_MINUTES} \
  python /kaggle/working/scripts/train_non_parametric_synthFCD.py fit \
    {ckpt_arg} \
    \
    --data.batch_size 2 \
    --data.num_workers 4 \
    --data.eval 0.2 \
    --data.fcd_intensity_range "[0.02, 0.3602]" \
    --data.fcd_tail_range "[14, 29]" \
    --data.split_seed 42 \
    --data.use_extra_data true \
    \
    --model.native_synthesis false \
    --model.flair_modality true \
    --model.seg_nb_levels 6 \
    --model.seg_features "[16,32,64,128,256,512]" \
    --model.time_limit_minutes {TRAINING_TIME_MINUTES} \
    --model.val_diagnostics_interval 10 \
    --model.debug_subject_ids '["sub-00001", "sub-00033", "sub-00044", "sub-00002", "sub-00058", "sub-00065"]' \
    \
    --trainer.default_root_dir {EXPERIMENTS_DIR} \
    --trainer.max_epochs 2000 \
    --trainer.accelerator gpu \
    --trainer.devices 1 \
    --trainer.precision 16-mixed \
    --trainer.enable_progress_bar false \
    --trainer.log_every_n_steps 5 \
    --trainer.num_sanity_val_steps 0 \
    \
    --checkpoint.save_top_k 1 \
    --checkpoint.monitor eval_loss \
    --checkpoint.mode min \
    --checkpoint.save_last true \
    \
    --seed_everything 0

---
## § 6 — Post-Training

Merges `metrics_history.csv` (all previous runs) with the newly written `metrics.csv` (current run) into a single complete `metrics.csv`. The `metrics_history.csv` is then deleted.

The output dataset will contain only `metrics.csv` — the complete training history across all runs.

In [ ]:
metrics_path = os.path.join(RUN_DIR, "metrics.csv")
history_path = os.path.join(RUN_DIR, "metrics_history.csv")

if os.path.exists(history_path) and os.path.exists(metrics_path):
    history_fields, history_rows = read_metric_csv(history_path)
    metrics_fields, metrics_rows = read_metric_csv(metrics_path)

    all_rows   = history_rows + metrics_rows
    fieldnames = history_fields + [c for c in metrics_fields if c not in history_fields]
    merged     = combine_metric_rows(all_rows)

    write_metric_csv(metrics_path, merged, fieldnames)
    os.remove(history_path)

elif os.path.exists(metrics_path):
    pass
else:
    print("[PostTraining] metrics.csv not found — training may not have produced output.")

---
## § 7 — Utilities

Helpers for manual operations. All cells are commented out by default.

In [ ]:
# ── Zip experiment directory ──────────────────────────────────────────────────
# shutil.make_archive('fcd_experiment_run', 'zip', EXPERIMENTS_DIR, RUN_NAME)

In [ ]:
# ── Wipe entire working directory (clean slate) ───────────────────────────────
# [shutil.rmtree(f.path, ignore_errors=True) if f.is_dir() else os.unlink(f.path)
#  for f in os.scandir("/kaggle/working/")]